# Задание 2 — Выбор алгоритма

**Задача:** бинарная классификация — предсказать факт ДТП на перекрёстке по признакам двух автомобилей.

Данные загружаются из 12 CSV-файлов сгенерированных в `task1.ipynb`.

In [124]:
from sklearn.neighbors import KNeighborsClassifier                                                                    
from sklearn.tree import DecisionTreeClassifier                                  
from sklearn.linear_model import LogisticRegression                                                                   
from sklearn.ensemble import RandomForestClassifier                              
from sklearn.model_selection import train_test_split, GridSearchCV                                                    
from sklearn.metrics import f1_score

import os
import pandas as pd
import time
import warnings
import joblib

warnings.filterwarnings('ignore') 

In [125]:
models = {
    "KNeibClass": KNeighborsClassifier(),
    "DecisionTreeClass": DecisionTreeClassifier(),
    "LogistRreg": LogisticRegression(),
    "RandForestClass": RandomForestClassifier()
}

## 1. Выбор моделей

Выбраны 4 классических метода:

| Модель | Обоснование |
|---|---|
| `KNeighborsClassifier` | Простой baseline, не требует обучения — хорошо показывает нижнюю границу качества |
| `DecisionTreeClassifier` | Интерпретируемая модель, способна точно восстановить кусочно-линейные правила — как наша функция `is_collision` |
| `LogisticRegression` | Быстрая линейная модель, устойчива на малых данных, хороший референс |
| `RandomForestClassifier` | Ансамбль деревьев — снижает дисперсию по сравнению с одним деревом, обычно лучший результат |

**Метрика: F1-score** — среднее гармоническое precision и recall. Выбрана потому что классы несбалансированы (~33% ДТП vs ~67% не-ДТП): accuracy в таком случае обманчива.

In [126]:
datasets = {}                                                                    
for f in os.listdir('datasets'):                                                                                      
    key = f.replace('.csv', '')                                                  
    datasets[key] = pd.read_csv(f'datasets/{f}')
    df = pd.read_csv(f'datasets/{f}')
    datasets[key] = pd.get_dummies(df)  

splits = {}                                                                                                           
for key, df in datasets.items():
    splits[key] = {                                                                                                   
        'X': df.drop(columns=['collision']),                                     
        'y': df['collision']                                                                                          
    }  

datasets["n5_s60"].head(10)

,car1_is_moving,car1_speed_level,car1_dist_to_intersection,car1_direction,car1_has_priority,car2_is_moving,car2_speed_level,car2_dist_to_intersection,car2_direction,car2_has_priority,collision,car1_car_type_motorcycle,car1_car_type_sedan,car1_car_type_suv,car1_car_type_truck,car2_car_type_motorcycle,car2_car_type_sedan,car2_car_type_suv,car2_car_type_truck
0,1,1,33.717229,1,1,0,0,25.440733,0,1,0,False,False,True,False,False,False,True,False
1,1,3,6.058430,3,0,1,3,32.743946,0,0,1,False,False,True,False,False,True,False,False
2,0,0,19.297831,0,1,0,0,17.025311,1,0,0,False,True,False,False,True,False,False,False
3,1,2,39.175837,1,1,1,2,27.824838,3,0,1,False,True,False,False,False,False,True,False
4,1,1,38.909653,0,0,1,3,34.155671,1,1,0,False,False,False,True,False,True,False,False
5,0,0,2.968027,2,1,0,0,38.821567,0,0,0,True,False,False,False,False,False,False,True
6,1,1,37.617632,2,0,1,3,24.629156,1,1,0,False,False,False,True,True,False,False,False
7,1,1,18.571087,3,0,1,1,38.279418,2,1,0,False,False,True,False,False,False,True,False
8,1,3,27.752570,0,0,1,2,38.899654,3,0,1,False,True,False,False,False,False,False,True
9,1,1,37.606302,3,0,1,3,38.325321,0,1,0,True,False,False,False,True,False,False,False


## 2. Загрузка датасетов

Номинальные признаки (`car_type`, `road_condition`) преобразуются через `pd.get_dummies()` — one-hot encoding.

In [127]:
results = {}

for key, split in splits.items():                                                                                     
    X_train, X_test, y_train, y_test = train_test_split(                         
        split['X'], split['y'], test_size=0.2, random_state=42                                                        
    )                                                                                                                 
    results[key] = {}                     
    for name, model in models.items():                                                                                
        model.fit(X_train, y_train)                                                                                   
        y_pred = model.predict(X_test)
        results[key][name] = f1_score(y_test, y_pred)   

pd.DataFrame(results).T 

,KNeibClass,DecisionTreeClass,LogistRreg,RandForestClass
n5_s750,0.626263,0.860215,0.815534,0.838095
n11_s750,0.250000,0.842105,0.734694,0.814159
n5_s60,0.333333,0.333333,0.285714,0.333333
n9_s750,0.266667,0.711111,0.744681,0.830189
n9_s60,0.000000,0.250000,0.285714,0.000000
n5_s300,0.595745,0.769231,0.772727,0.808511
n9_s1500,0.389937,0.854167,0.798030,0.870813
n11_s300,0.387097,0.812500,0.583333,0.785714
n11_s60,0.000000,0.363636,0.800000,0.285714
n11_s1500,0.347826,0.814070,0.800000,0.852018


## 3. Обучение на 12 датасетах

In [ ]:
df_results = pd.DataFrame(results).T

# Среднее по всем датасетам
print("=== Среднее F1 по всем датасетам ===")
print(df_results.mean().sort_values(ascending=False).to_string())

# Упор на малые датасеты (60 строк)
small_keys = [k for k in df_results.index if 's60' in k]
print("\n=== Среднее F1 на малых датасетах (n=60) ===")
print(df_results.loc[small_keys].mean().sort_values(ascending=False).to_string())

In [129]:
param_grids = {                                                                                                       
    "DecisionTreeClass": {             
        'max_depth': [3, 5, 10, None],                                                                                
        'min_samples_split': [2, 5, 10]                                          
    },                                                                                                                
    "LogistRreg": {                                                              
        'C': [0.01, 0.1, 1, 10],
        'max_iter': [1000, 3000, 5000]                                                                                            
    },                           
    "RandForestClass": {                                                                                              
        'n_estimators': [10, 50, 100],                                           
        'max_depth': [3, 5, None]                                                                                     
    }                   
}      

## 4. Выбор 3 лучших моделей

По результатам таблицы выбываем `KNeighborsClassifier` — наихудший F1, особенно критично на малых данных (нули). Оставляем: `DecisionTreeClassifier`, `LogisticRegression`, `RandomForestClassifier`.

## 5. Подбор гиперпараметров (GridSearchCV)

GridSearchCV перебирает все комбинации параметров через 5-фолдовую кросс-валидацию.  
Замеряем время на **самом маленьком** (`n5_s60`) и **самом большом** (`n11_s1500`) датасете.

In [130]:
for dataset_name, split in [('small (n5_s60)', splits['n5_s60']), ('large (n11_s1500)', splits['n11_s1500'])]:        
    X_train, X_test, y_train, y_test = train_test_split(                         
        split['X'], split['y'], test_size=0.2, random_state=42                                                        
    )                                                                                                                 
    print(f"\n=== {dataset_name} ===")                                                                                
    for name, params in param_grids.items():                                                                          
        grid = GridSearchCV(models[name], params, scoring='f1', cv=5)            
        start = time.time()                                                                                           
        grid.fit(X_train, y_train)                                               
        elapsed = time.time() - start                                                                                 
        print(f"{name}: {elapsed:.2f}s, best={grid.best_params_}, f1={grid.best_score_:.3f}")



=== small (n5_s60) ===
DecisionTreeClass: 0.12s, best={'max_depth': 10, 'min_samples_split': 10}, f1=0.683
LogistRreg: 0.26s, best={'C': 10, 'max_iter': 1000}, f1=0.578
RandForestClass: 0.89s, best={'max_depth': None, 'n_estimators': 10}, f1=0.738

=== large (n11_s1500) ===
DecisionTreeClass: 0.23s, best={'max_depth': 5, 'min_samples_split': 2}, f1=0.862
LogistRreg: 1.60s, best={'C': 1, 'max_iter': 1000}, f1=0.816
RandForestClass: 1.54s, best={'max_depth': None, 'n_estimators': 100}, f1=0.850


In [131]:
df = splits['n5_s1500']
X_train, X_test, y_train, y_test = train_test_split(                         
        df['X'], df['y'], test_size=0.2, random_state=42                                                        
    ) 

mod = RandomForestClassifier(max_depth=None, n_estimators=100)  
mod.fit(X_train,y_train)
y_pred = mod.predict(X_test)
f1 = f1_score(y_test, y_pred)
print("RandForestClass:", f1)

RandForestClass: 0.8956521739130435


## 6. Сохранение лучшей модели

Сохраняем `RandomForestClassifier` с лучшими гиперпараметрами из GridSearchCV, обученный на самом большом датасете (`n5_s1500` — 1500 строк, 5 признаков на объект).

In [132]:
joblib.dump(mod, 'model.pkl')

['model.pkl']